In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

In [2]:
df = pd.read_csv("train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))].copy()
df['TotalSF'] = df['TotalBsmtSF'].fillna(0)+df['1stFlrSF']+ df['2ndFlrSF']
df['HouseAge'] = df['YrSold'] - df['YearBuilt']
df['TotalBath'] = (
    df['FullBath']
    + 0.5 * df['HalfBath']
    + df['BsmtFullBath'].fillna(0)
    + 0.5 * df['BsmtHalfBath'].fillna(0)
)


In [4]:
selected_cols = [
    'LotArea', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'FullBath',
    'HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'GarageCars',
    'OverallQual', 'OverallCond', 'YearBuilt',
    'TotalSF', 'HouseAge', 'TotalBath', 'GarageArea', 'YearRemodAdd'
]
X = df[selected_cols].copy()
X = X.fillna(X.mean())


y = np.log1p(df['SalePrice'].copy())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
}

best_r2 = -1
best_model_name = ""
best_model_obj = None

print("\n2. Sikl (For) boshlandi. Modellar o'qitilmoqda...\n")
print(f"{'Model Nomi':<20} | {'MAE (Xatolik)':<16} | {'RMSE':<16} | {'R2 Score (Aniqlik)'}")
print("-" * 75)


2. Sikl (For) boshlandi. Modellar o'qitilmoqda...

Model Nomi           | MAE (Xatolik)    | RMSE             | R2 Score (Aniqlik)
---------------------------------------------------------------------------


In [6]:
for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = np.expm1(model.predict(X_test))
    y_true = np.expm1(y_test)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) * 100  # Foizga o'tkazamiz

    print(f"{name:<20} | ${mae:<15,.2f} | ${rmse:<15,.2f} | {r2:.2f}%")

    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name
        best_model_obj = model

print("-" * 75)
print(f"\n🏆 ENG YAXSHI MODEL: {best_model_name} ({best_r2:.2f}%)")


Linear Regression    | $17,633.16       | $24,170.49       | 89.42%
Ridge Regression     | $17,621.26       | $24,168.83       | 89.43%
Decision Tree        | $24,740.35       | $37,355.01       | 74.74%
Random Forest        | $16,814.80       | $24,406.97       | 89.22%
Gradient Boosting    | $15,460.84       | $21,630.46       | 91.53%
---------------------------------------------------------------------------

🏆 ENG YAXSHI MODEL: Gradient Boosting (91.53%)


In [7]:
print("Eng yaxshi model saqlanmoqda...")
os.makedirs("models", exist_ok = True)

joblib.dump(best_model_obj, "models/protech_model.joblib")
joblib.dump(selected_cols, 'models/protech_features.joblib')

print(f"--- '{best_model_name}' modeli 'models/proptech_model.joblib' nomi bilan saqlandi.")
print("--- Eslatma: bashorat log-narxda qilingani uchun, backendda natijani")
print("--- np.expm1(model.predict(X)) orqali haqiqiy dollarga aylantiring.")


Eng yaxshi model saqlanmoqda...
--- 'Gradient Boosting' modeli 'models/proptech_model.joblib' nomi bilan saqlandi.
--- Eslatma: bashorat log-narxda qilingani uchun, backendda natijani
--- np.expm1(model.predict(X)) orqali haqiqiy dollarga aylantiring.
